# NSAI Level 1C — Symbol-Aligned Neuro-Symbolic Intent Classification

**Architecture:** Neural Detectors → Symbolization → Pure Symbolic Rules

- Train 1 binary detector per intent (TF-IDF + Logistic Regression)
- Detectors output probabilities internally
- Symbolization converts numeric outputs → symbolic predicates
- Rule engine operates ONLY on symbols (Kautz-aligned Level-1)

Dataset: `utterance`, `intent` (4 classes)

## 1. Imports

Standard scientific and ML libraries required for training detectors, running evaluation, and displaying results throughout the notebook.

In [29]:
import pandas as pd
import numpy as np
import json
from collections import Counter, defaultdict

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

## 2. Module Reload

Hot-reloads `level1c_model` so any edits made to the source file take effect in the current kernel session without requiring a full restart. Also imports the `Level1CClassifier` class, the default `CONFIG` dict (numeric thresholds only), and the ordered `INTENTS` list.

In [30]:
import importlib
import level1c_model
importlib.reload(level1c_model)

from level1c_model import Level1CClassifier, CONFIG, INTENTS

## 3. Load and Validate Dataset

Reads the shared intent dataset from `data/intents_base.csv`. Asserts that the file contains exactly the expected columns (`utterance`, `intent`), normalises intent labels to lowercase/stripped, and prints the per-class sample counts as a quick sanity check before splitting.

In [31]:
df = pd.read_csv("../data/intents_base.csv")

assert set(df.columns) == {"utterance", "intent"}, f"Expected columns {{utterance, intent}}, got {set(df.columns)}"

df["intent"] = df["intent"].str.lower().str.strip()

print("Class Distribution:")
print(df["intent"].value_counts())
print(f"\nTotal: {len(df)} records")

Class Distribution:
intent
out_of_scope     480
execution        413
investigate      412
summarization    356
Name: count, dtype: int64

Total: 1661 records


## 4. Train / Test Split (Stratified)

Splits the dataset 80 / 20 with `stratify=y` so every intent class is proportionally represented in both partitions. A fixed `random_state` ensures reproducibility across runs.

In [32]:
X = df["utterance"]
y = df["intent"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")

Train: 1328 | Test: 333


## 5. Train Binary Detectors

One TF-IDF + Logistic Regression pipeline is trained **per intent** as a binary classifier ("is this utterance of this intent?"). Detectors are independent — their scores do not sum to 1, so the symbolization layer can reason over all four scores simultaneously without assuming a probability simplex.

In [33]:
intents = INTENTS
detectors = {}

print("Training binary detectors...\n")

for intent in intents:
    y_binary_train = (y_train == intent).astype(int)
    y_binary_test = (y_test == intent).astype(int)

    detector = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=5000,
            ngram_range=(1, 2),
            stop_words="english"
        )),
        ("clf", LogisticRegression(
            solver="lbfgs",
            random_state=42,
            max_iter=1000
        ))
    ])

    detector.fit(X_train, y_binary_train)

    # quick sanity on detector behavior
    acc = detector.score(X_test, y_binary_test)

    detectors[intent] = detector
    print(f"{intent:>15} detector: Accuracy={acc:.4f}")

print(f"\nAll {len(detectors)} detectors trained")

Training binary detectors...

    investigate detector: Accuracy=0.9189
      execution detector: Accuracy=0.8769
  summarization detector: Accuracy=0.9820
   out_of_scope detector: Accuracy=0.8589

All 4 detectors trained


## 6. Assemble the Level 1C Classifier

Wraps all four binary detectors into a `Level1CClassifier`. At inference time it composes three layers:

1. **Neural layer** — each detector returns a probability score (numeric, internal only)
2. **Symbolization layer** — scores + raw/unique/model token counts are converted to symbolic predicates (`CANDIDATE_*`, `HIGH_CONFIDENCE_*`, `AMBIGUOUS`, `*_TOKEN_COUNT_*`, etc.); all numeric thresholds live _only_ here
3. **Rule engine** — operates exclusively on the predicate set; rules are loaded from `rules.json` and contain no numeric comparisons

In [34]:
classifier = Level1CClassifier(detectors=detectors, intents=intents)

print("Level 1C classifier initialized")
print("Config:")
print(json.dumps(CONFIG, indent=2))
print("Intents:", intents)

Level 1C classifier initialized
Config:
{
  "BASE_MIN_SCORE": 0.3,
  "HIGH_CONFIDENCE_SCORE": 0.85,
  "AMBIGUITY_MARGIN": 0.1,
  "RAW_MIN_TOKENS": 2,
  "MODEL_MIN_TOKENS": 2,
  "UNIQUE_MIN_TOKENS": 2
}
Intents: ['investigate', 'execution', 'summarization', 'out_of_scope']


## 7. Qualitative Spot-Check

Runs inference on a small set of hand-picked utterances covering all intent classes and likely decision paths. For each utterance the full predicate set (`symbols`) and the rule-driven decision (`decision_state`, `decision_reason`, `triggered_rules`) are printed — useful for verifying that the symbolization layer and rule engine behave as expected before running the full evaluation.

In [35]:
test_utterances = [
    "why is server cpu high",
    "restart nginx on host123",
    "summarize the incident from yesterday",
    "hello",
    "delete all production databases",
    "server issues yesterday"
]

for u in test_utterances:
    out = classifier.predict(u)
    print("\n" + "="*90)
    print("UTTERANCE:", u)
    print("- symbols:", sorted(out["symbols"]))
    print("- predicted_intent:", out["predicted_intent"])
    print("- decision_state:", out["decision_state"])
    print("- decision_reason:", out["decision_reason"])
    print("- triggered_rules:", out["triggered_rules"])


UTTERANCE: why is server cpu high
- symbols: ['CANDIDATE_INVESTIGATE', 'MODEL_TOKEN_COUNT_SUFFICIENT', 'NOT_HIGH_CONFIDENCE_EXECUTION', 'RAW_TOKEN_COUNT_SUFFICIENT', 'UNIQUE_TOKEN_COUNT_SUFFICIENT']
- predicted_intent: investigate
- decision_state: accepted
- decision_reason: R_DEFAULT
- triggered_rules: ['R_DEFAULT']

UTTERANCE: restart nginx on host123
- symbols: ['CANDIDATE_EXECUTION', 'MODEL_TOKEN_COUNT_SUFFICIENT', 'NOT_HIGH_CONFIDENCE_EXECUTION', 'RAW_TOKEN_COUNT_SUFFICIENT', 'UNIQUE_TOKEN_COUNT_SUFFICIENT']
- predicted_intent: execution
- decision_state: needs_clarification
- decision_reason: R_EXECUTION_LOW_CONFIDENCE
- triggered_rules: ['R_EXECUTION_LOW_CONFIDENCE']

UTTERANCE: summarize the incident from yesterday
- symbols: ['CANDIDATE_SUMMARIZATION', 'MODEL_TOKEN_COUNT_SUFFICIENT', 'NOT_HIGH_CONFIDENCE_EXECUTION', 'RAW_TOKEN_COUNT_SUFFICIENT', 'UNIQUE_TOKEN_COUNT_SUFFICIENT']
- predicted_intent: summarization
- decision_state: accepted
- decision_reason: R_DEFAULT
- trigge

## 8. Full Test-Set Evaluation

Runs inference over the entire held-out test set and aggregates:

- **Intent accuracy** — fraction of utterances where `predicted_intent == true_intent`
- **Decision state distribution** — how many utterances were `accepted`, `needs_clarification`, or `blocked`
- **Rule trigger counts** — how frequently each symbolic rule (and override branch) fired, useful for detecting over-triggering

In [36]:
pred_intents = []
decision_states = []
decision_reasons = []
rule_counts = Counter()

for text in X_test:
    out = classifier.predict(text)
    pred_intents.append(out["predicted_intent"])
    decision_states.append(out["decision_state"])
    decision_reasons.append(out["decision_reason"])
    for r in out["triggered_rules"]:
        rule_counts[r] += 1

accuracy = np.mean([p == t for p, t in zip(pred_intents, y_test)])
print(f"Accuracy (intent only): {accuracy:.4f}")

print("\nDecision State Distribution:")
state_counts = Counter(decision_states)
for k, v in state_counts.items():
    print(f"  {k:>20}: {v:3d} ({v/len(X_test)*100:.1f}%)")

print("\nRule Trigger Counts:")
for rule, cnt in rule_counts.most_common():
    print(f"  {rule:>30}: {cnt:3d} ({cnt/len(X_test)*100:.1f}%)")

Accuracy (intent only): 0.9309

Decision State Distribution:
               blocked:  14 (4.2%)
   needs_clarification:  85 (25.5%)
              accepted: 234 (70.3%)

Rule Trigger Counts:
                       R_DEFAULT: 232 (69.7%)
      R_EXECUTION_LOW_CONFIDENCE:  81 (24.3%)
           R_NO_CANDIDATE_INTENT:  14 (4.2%)
                     R_AMBIGUOUS:   6 (1.8%)
  R_EXECUTION_LOW_CONFIDENCE::override:   2 (0.6%)


## 9. Per-Class Classification Report

Scikit-learn `classification_report` broken down by intent class — precision, recall, F1-score, and support. Identifies which intents are hardest for the detector ensemble to classify correctly.

In [37]:
print(classification_report(y_test, pred_intents))

               precision    recall  f1-score   support

    execution       0.90      0.89      0.90        83
  investigate       0.97      0.86      0.91        83
 out_of_scope       0.88      0.99      0.93        96
summarization       1.00      0.99      0.99        71

     accuracy                           0.93       333
    macro avg       0.94      0.93      0.93       333
 weighted avg       0.93      0.93      0.93       333



## 10. Confusion Matrix

Raw count confusion matrix across the four intent classes. Off-diagonal cells reveal systematic misclassification patterns (e.g., `investigate` predicted when `execution` is true), guiding targeted rule or detector improvements.

In [38]:
labels = intents
cm = confusion_matrix(y_test, pred_intents, labels=labels)

print("Labels:", labels)
print(cm)

Labels: ['investigate', 'execution', 'summarization', 'out_of_scope']
[[71  7  0  5]
 [ 2 74  0  7]
 [ 0  0 70  1]
 [ 0  1  0 95]]


## 11. Symbol Frequency Analysis

Counts how often each symbolic predicate appears across the test set, both overall and broken down by true intent class. Surfaces dominant predicates, uneven predicate coverage across intents, and any unexpected symbols that might indicate drift in the symbolization layer.

In [43]:
symbol_counts = Counter()
symbol_by_intent = defaultdict(Counter)

for text, true_intent in zip(X_test, y_test):
    out = classifier.predict(text)
    for s in out["symbols"]:
        symbol_counts[s] += 1
        symbol_by_intent[true_intent][s] += 1

print("Top Symbols Overall:")
for s, c in symbol_counts.most_common(25):
    print(f"{s:>30}: {c} ({c/len(X_test)*100:.1f}% of test samples)")

Top Symbols Overall:
    RAW_TOKEN_COUNT_SUFFICIENT: 333 (100.0% of test samples)
 UNIQUE_TOKEN_COUNT_SUFFICIENT: 333 (100.0% of test samples)
 NOT_HIGH_CONFIDENCE_EXECUTION: 330 (99.1% of test samples)
  MODEL_TOKEN_COUNT_SUFFICIENT: 261 (78.4% of test samples)
        CANDIDATE_OUT_OF_SCOPE: 97 (29.1% of test samples)
           CANDIDATE_EXECUTION: 84 (25.2% of test samples)
         CANDIDATE_INVESTIGATE: 73 (21.9% of test samples)
MODEL_TOKEN_COUNT_INSUFFICIENT: 72 (21.6% of test samples)
       CANDIDATE_SUMMARIZATION: 70 (21.0% of test samples)
                     AMBIGUOUS: 34 (10.2% of test samples)
  HIGH_CONFIDENCE_OUT_OF_SCOPE: 26 (7.8% of test samples)
           NO_CANDIDATE_INTENT: 14 (4.2% of test samples)
 HIGH_CONFIDENCE_SUMMARIZATION: 9 (2.7% of test samples)
   HIGH_CONFIDENCE_INVESTIGATE: 6 (1.8% of test samples)
     HIGH_CONFIDENCE_EXECUTION: 3 (0.9% of test samples)


## 12. Error and Success Case Inspection

Collects all misclassified and correctly classified test utterances. Each entry includes the full predicate set and the complete rule path that produced the decision. Reviewing errors helps identify correctable patterns — e.g., a rule firing too broadly, a weak detector for a specific phrasing, or a missing predicate.

In [ ]:
errors = []
for text, true_intent, pred_intent in zip(X_test, y_test, pred_intents):
    if true_intent != pred_intent:
        out = classifier.predict(text)
        errors.append({
            "utterance": text,
            "true_intent": true_intent,
            "predicted_intent": pred_intent,
            "decision_state": out["decision_state"],
            "decision_reason": out["decision_reason"],
            "triggered_rules": out["triggered_rules"],
            "symbols": sorted(out["symbols"])
        })

print(f"Total errors: {len(errors)} / {len(X_test)}")
errors[:10]

#print sucess cases as well
successes = []
for text, true_intent, pred_intent in zip(X_test, y_test, pred_intents):
    if true_intent == pred_intent:
        out = classifier.predict(text)
        successes.append({
            "utterance": text,
            "true_intent": true_intent,
            "predicted_intent": pred_intent,
            "decision_state": out["decision_state"],
            "decision_reason": out["decision_reason"],
            "triggered_rules": out["triggered_rules"],
            "symbols": sorted(out["symbols"])
        })
print(f"Total successes: {len(successes)} / {len(X_test)}")
successes[:10]

Total errors: 23 / 333


## 13. Persist Model

Serialises the trained detector pipelines (one `.pkl` per intent), the `config.json` (numeric thresholds and intents list), and the symbolic rule set as a human-readable `rules.json` to `models/level1c/`. Rules are stored as plain JSON data — independent of the Python source — so they can be inspected, versioned, and modified without code changes.

In [41]:
model_dir = "../models/level1c"
classifier.save(model_dir)

✓ Saved Level1C model to ..\models\level1c


## 14. Save Evaluation Artifacts

Writes the full test-set predictions (one row per utterance, including symbols and triggered rules) to `artifacts/level1c/level1c_predictions.csv`, and a JSON summary of accuracy, decision-state counts, rule trigger counts, and the config snapshot to `artifacts/level1c/evaluation_metrics.json`. Both files are intended for downstream consumption by Level 2+ or for regression tracking between runs.

In [42]:
import os
from datetime import datetime

artifacts_dir = "../artifacts/level1c"
os.makedirs(artifacts_dir, exist_ok=True)

# Save predictions
rows = []
for i, (text, true_intent) in enumerate(zip(X_test, y_test)):
    out = classifier.predict(text)
    rows.append({
        "test_index": i,
        "utterance": text,
        "true_intent": true_intent,
        "predicted_intent": out["predicted_intent"],
        "decision_state": out["decision_state"],
        "decision_reason": out["decision_reason"],
        "triggered_rules": ",".join(out["triggered_rules"]),
        "symbols": ",".join(sorted(out["symbols"]))
    })

df_pred = pd.DataFrame(rows)
pred_path = os.path.join(artifacts_dir, "level1c_predictions.csv")
df_pred.to_csv(pred_path, index=False)

# Save metrics
metrics = {
    "model": "level1c_symbol_aligned",
    "timestamp": datetime.now().isoformat(),
    "test_size": len(X_test),
    "accuracy": float(accuracy),
    "decision_state_distribution": dict(state_counts),
    "rule_trigger_counts": dict(rule_counts),
    "config": CONFIG,
    "intents": intents
}
metrics_path = os.path.join(artifacts_dir, "evaluation_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("Saved:")
print(" -", pred_path)
print(" -", metrics_path)

Saved:
 - ../artifacts/level1c\level1c_predictions.csv
 - ../artifacts/level1c\evaluation_metrics.json
